# BPE
A Byte-Pair Encoding (BPE) tokenizer is a subword tokenization algorithm used in Natural Language Processing (NLP) and Large Language Models (LLMs) to convert text into numerical token IDs. Originally a data compression algorithm introduced by Philip Gage in 1994, BPE balances vocabulary size with the ability to represent rare or unseen words by breaking text into common subwords.

### UTF-8 encoding

In [162]:
text = """Byte-Pair Encoding (BPE) is a widely used subword tokenization algorithm in natural language processing
and large language models that bridges the gap between character-level and word-level text representation by iteratively merging the most frequent adjacent symbols into longer subword units.
Originally created by Philip Gage in 1994 as a general data compression technique, BPE was adapted for neural machine translation and modern transformers to keep vocabulary sizes manageable while eliminating out-of-vocabulary failures.
During training, the algorithm begins with a base vocabulary—typically individual characters or, in byte-level BPE, all 256 possible UTF-8 bytes—and splits a training corpus into those base units before repeatedly scanning the data to 
count every adjacent pair of tokens, merging the single most frequent pair into a brand-new token, adding that token to the vocabulary, and recording the merge rule until a predetermined vocabulary size is reached. Once trained, the 
tokenizer processes new text by breaking it down into the same base characters or bytes and greedily applying the learned merge rules in the exact priority order they were discovered, allowing frequent words like "playing" to remain single 
tokens while rare, misspelled, or compound words are cleanly decomposed into meaningful prefixes, roots, suffixes, or raw bytes so the model can still process any Unicode character, emoji, or code snippet without losing information."""
print(text)

Byte-Pair Encoding (BPE) is a widely used subword tokenization algorithm in natural language processing
and large language models that bridges the gap between character-level and word-level text representation by iteratively merging the most frequent adjacent symbols into longer subword units.
Originally created by Philip Gage in 1994 as a general data compression technique, BPE was adapted for neural machine translation and modern transformers to keep vocabulary sizes manageable while eliminating out-of-vocabulary failures.
During training, the algorithm begins with a base vocabulary—typically individual characters or, in byte-level BPE, all 256 possible UTF-8 bytes—and splits a training corpus into those base units before repeatedly scanning the data to 
count every adjacent pair of tokens, merging the single most frequent pair into a brand-new token, adding that token to the vocabulary, and recording the merge rule until a predetermined vocabulary size is reached. Once trained, the 

In [163]:
byteTokens = text.encode("utf-8")
byteTokens = list(byteTokens)
byteTokens[:20] # Max vocab size= 256

[66,
 121,
 116,
 101,
 45,
 80,
 97,
 105,
 114,
 32,
 69,
 110,
 99,
 111,
 100,
 105,
 110,
 103,
 32,
 40]

In [164]:
print(f"Tokens= {len(byteTokens)}")

Tokens= 1478


### Finding the frequency of character combination pairs

##### Function to count pairwise token frequincy

In [165]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [166]:
def ex_stats(text): # Example
    counts = {}
    for pair in zip(text, text[1:]):
        pair = "".join(pair)
        counts[pair] = counts.get(pair, 0) + 1
    return counts
damoText = """Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
 accusantium doloremque laudantium, totam rem aperiam Lorem Lorem Lorem Lorem Lorem Lorem Lorem, 
 tum deleniti atque corrupti quos dolores et quas molestias excepturi sint occaecati cupiditat"""
ex_stats(damoText)

{'Lo': 8,
 'or': 19,
 're': 17,
 'em': 11,
 'm ': 17,
 ' i': 7,
 'ip': 3,
 'ps': 1,
 'su': 2,
 'um': 6,
 ' d': 9,
 'do': 8,
 'ol': 9,
 'lo': 6,
 'r ': 5,
 ' s': 4,
 'si': 4,
 'it': 8,
 't ': 18,
 ' a': 10,
 'am': 5,
 'me': 1,
 'et': 4,
 't,': 2,
 ', ': 5,
 ' c': 8,
 'co': 5,
 'on': 4,
 'ns': 2,
 'se': 4,
 'ec': 3,
 'ct': 1,
 'te': 5,
 'tu': 4,
 'ur': 5,
 'ad': 2,
 'di': 3,
 'pi': 3,
 'is': 5,
 'sc': 1,
 'ci': 5,
 'in': 8,
 'ng': 1,
 'g ': 1,
 ' e': 12,
 'el': 3,
 'li': 5,
 't.': 2,
 '. ': 4,
 ' S': 1,
 'Se': 1,
 'ed': 1,
 'd ': 5,
 'o ': 3,
 'ei': 1,
 'iu': 3,
 'us': 2,
 'sm': 1,
 'mo': 4,
 'od': 2,
 ' t': 3,
 'mp': 1,
 'po': 1,
 'nc': 1,
 'id': 6,
 'du': 1,
 'un': 3,
 'nt': 8,
 ' u': 3,
 'ut': 3,
 ' l': 4,
 'la': 6,
 'ab': 3,
 'bo': 3,
 'e ': 9,
 ' m': 4,
 'ma': 1,
 'ag': 1,
 'gn': 1,
 'na': 1,
 'a ': 5,
 'al': 2,
 'iq': 2,
 'qu': 9,
 'ua': 3,
 'a.': 1,
 ' U': 1,
 'Ut': 1,
 'en': 5,
 'ni': 6,
 'im': 3,
 'mi': 1,
 ' v': 3,
 've': 2,
 'ia': 6,
 'm,': 3,
 ' q': 4,
 'ui': 4,
 's ': 7,
 ' 

In [167]:
get_stats(byteTokens), f"Token size reduced from {len(byteTokens)} to {len(get_stats(byteTokens))} by just pairing."

({(66, 121): 1,
  (121, 116): 5,
  (116, 101): 17,
  (101, 45): 2,
  (45, 80): 1,
  (80, 97): 1,
  (97, 105): 8,
  (105, 114): 3,
  (114, 32): 11,
  (32, 69): 1,
  (69, 110): 1,
  (110, 99): 2,
  (99, 111): 10,
  (111, 100): 6,
  (100, 105): 6,
  (105, 110): 39,
  (110, 103): 22,
  (103, 32): 13,
  (32, 40): 1,
  (40, 66): 1,
  (66, 80): 3,
  (80, 69): 3,
  (69, 41): 1,
  (41, 32): 1,
  (32, 105): 15,
  (105, 115): 4,
  (115, 32): 23,
  (32, 97): 22,
  (97, 32): 8,
  (32, 119): 10,
  (119, 105): 4,
  (105, 100): 3,
  (100, 101): 9,
  (101, 108): 9,
  (108, 121): 8,
  (121, 32): 17,
  (32, 117): 4,
  (117, 115): 2,
  (115, 101): 8,
  (101, 100): 13,
  (100, 32): 15,
  (32, 115): 14,
  (115, 117): 3,
  (117, 98): 2,
  (98, 119): 2,
  (119, 111): 5,
  (111, 114): 20,
  (114, 100): 7,
  (32, 116): 32,
  (116, 111): 15,
  (111, 107): 6,
  (107, 101): 8,
  (101, 110): 14,
  (110, 105): 11,
  (105, 122): 4,
  (122, 97): 1,
  (97, 116): 13,
  (116, 105): 8,
  (105, 111): 6,
  (111, 110): 6,
  